# Background — Python Basics (อ่านเองก่อนเข้าโมดูลถัดไป)

โน้ตบุ๊กนี้เป็น "ฟื้นพื้นฐาน Python" สำหรับคนที่เคยเขียนมาบ้าง หรือเพิ่งเริ่ม — วัตถุประสงค์ไม่ใช่สอน Python ครบทุกมิติ แต่ให้คุณ **อ่านโค้ดในหลักสูตรออกและแก้เองได้** ถ้าส่วนไหนถนัดอยู่แล้ว ข้ามได้เลย

## 1. ตัวแปรและชนิดข้อมูลที่ใช้จริงในงานข้อมูล

In [1]:
# ชนิดพื้นฐานที่จะเจอทุกวัน
load_g = 100.0          # float — ค่าที่วัดได้เสมอเป็น float
n_readings = 40         # int — จำนวนนับ
sensor_id = "LC-01"     # str  — ชื่อ/ป้ายกำกับ
is_calibrated = True    # bool — สถานะ

print(load_g, n_readings, sensor_id, is_calibrated)
print(type(load_g), type(n_readings), type(sensor_id), type(is_calibrated))

100.0 40 LC-01 True
<class 'float'> <class 'int'> <class 'str'> <class 'bool'>


**จุดที่มือใหม่พลาดบ่อย:** `100` กับ `100.0` ต่างกัน — int กับ float การคำนวณเมโทรโลยีใช้ float เกือบทั้งหมด และอย่าลืมว่า float มีความละเอียดจำกัด (ไม่ใช่ค่าจริงทศนิยมไม่รู้จบ)

In [2]:
print(100 == 100.0)        # เทียบค่าได้: True
print(0.1 + 0.2)           # แต่ 0.1 + 0.2 ไม่เท่า 0.3 เป๊ะ ๆ — เรื่องที่ต้องจำในงานวัด!

True
0.30000000000000004


## 2. f-string — จัดฟอร์แมตผลลัพธ์ให้อ่านง่าย

In [3]:
mean = 24.73186
std = 0.05231
n = 40

print(f"n = {n} readings, mean = {mean:.2f} °C, std = {std:.3f} °C")
print(f"uncertainty (2σ) = {2 * std:.3f} °C")

n = 40 readings, mean = 24.73 °C, std = 0.052 °C
uncertainty (2σ) = 0.105 °C


## 3. คอลเลกชัน: list, dict และการวนลูป

In [4]:
channels = ["ch1", "ch2", "ch3"]          # list — ลำดับ ซ้ำได้
offsets = {"ch1": 0.02, "ch2": -0.05, "ch3": 0.01}   # dict — คู่ key: value

# วนลูป list
for ch in channels:
    print(ch, "->", offsets[ch])

# list comprehension: สร้าง list ใหม่จากเงื่อนไข — จะเจอบ่อยมากในโค้ด ML
big = [c for c, v in offsets.items() if abs(v) >= 0.02]
print("channels needing adjustment:", big)

ch1 -> 0.02
ch2 -> -0.05
ch3 -> 0.01
channels needing adjustment: ['ch1', 'ch2']


## 4. ฟังก์ชัน — ห่อการคำนวณซ้ำ ๆ ให้เรียกใช้ได้หลายครั้ง

In [5]:
def to_celsius(temp_f: float) -> float:
    """แปลงองศาฟาเรนไฮต์เป็นเซลเซียส."""
    return (temp_f - 32) * 5 / 9


print(to_celsius(68.0), to_celsius(32.0))

# ค่าดีฟอลต์ให้พารามิเตอร์ — ลดจำนวนอาร์กิวเมนต์ที่ต้องส่งทุกครั้ง
def zscore(values: list[float], mean: float | None = None) -> list[float]:
    """แปลงเป็นคะแนนมาตรฐาน (z) — ถ้าไม่ส่ง mean มา คำนวณเอง."""
    if mean is None:
        mean = sum(values) / len(values)
    std = (sum((v - mean) ** 2 for v in values) / len(values)) ** 0.5
    return [(v - mean) / std for v in values]


print([f"{z:.2f}" for z in zscore([10.0, 12.0, 14.0, 16.0])])

20.0 0.0
['-1.34', '-0.45', '0.45', '1.34']


## 5. import — ความสามารถเกือบทั้งหมดของ Python มาจากไลบรารี

In [6]:
import math

import numpy as np          # ธรรมเนียมการตั้งชื่อย่อที่ทั้งโลกใช้เหมือนกัน

print(math.sqrt(2), np.sqrt(np.array([2.0, 4.0, 8.0])))

1.4142135623730951 [1.41421356 2.         2.82842712]


## 6. NumPy เร่งเรียง — สิ่งที่อยู่ใต้ไลบรารี ML ทุกตัว

NumPy array = กล่องตัวเลขหลายช่องที่ **คำนวณทั้งกล่องพร้อมกันได้** (vectorized) — สาเหตุที่โค้ด ML เขียนสั้นและเร็ว

In [7]:
import numpy as np

readings = np.array([24.1, 24.3, 24.0, 24.2, 24.1])

print("mean:", readings.mean())
print("std :", readings.std())
print("deviation from mean:", readings - readings.mean())   # คำนวณทั้ง array ในคำสั่งเดียว

# slicing เหมือน list: [start:stop] — stop ไม่ถูกรวม
print("first two:", readings[:2])

# สร้างข้อมูลจำลองด้วย seed — เทคนิคสำคัญ: ใส่ seed ทุกครั้งเพื่อให้ทำซ้ำได้
rng = np.random.default_rng(42)
noise = rng.normal(loc=0.0, scale=0.5, size=5)
print("seeded noise:", noise)

mean: 24.140000000000004
std : 0.10198039027185561
deviation from mean: [-0.04  0.16 -0.14  0.06 -0.04]
first two: [24.1 24.3]
seeded noise: [ 0.15235854 -0.51999205  0.3752256   0.47028236 -0.97551759]


## 7. อ่าน error message ให้เป็น — ทักษะที่ประหยัดเวลาที่สุด

Traceback อ่านจาก **ล่างขึ้นบน**: บรรทัดสุดท้ายบอกว่า *เกิดอะไร* บรรทัดก่อนหน้าบอกว่า *เกิดที่ไหน*

In [8]:
# ลองรันแล้วอ่าน error — แก้ถูกไหม? บรรทัดสุดท้ายบอกว่า list index out of range
values = [10.0, 20.0, 30.0]
print(values[3])   # index เริ่มที่ 0 ดังนั้นช่องสุดท้ายคือ [2]

IndexError: list index out of range

## เช็กตัวเอง

1. `0.1 + 0.2 == 0.3` ได้ค่าอะไร เพราะอะไร? แล้วถ้าต้องเทียบ float จริง ๆ ควรทำอย่างไร? (คำใบ้: `math.isclose`)
2. เขียนฟังก์ชัน `cv(mean, std)` คืนค่า coefficient of variation = std/mean
3. สร้าง array 10 ค่าจาก `rng.normal(100, 2, 10)` ด้วย seed เดิมสองครั้ง — ค่าเท่ากันไหม เพราะอะไร?

เฉลยอยู่ใน `exercise.ipynb` และ `solution.ipynb` ของโฟลเดอร์นี้